# ZEPTO DATA & AI PLATFORM

## Module 1 — Data Pipeline

This module implements a complete data pipeline for collecting, cleaning, transforming, storing, and analyzing book data.

### Pipeline

Web Scraping → Data Cleaning → Currency Conversion → SQLite Database → SQL Analysis → Pandas Analysis

### Dataset

The pipeline collects book information including:

- Title
- Category
- Price in GBP
- Price in INR
- Rating
- Stock availability

The final dataset contains 60+ books across 3+ categories.

# New Section

IMPORTS


In [5]:
import requests
import pandas as pd
import sqlite3
import time

from bs4 import BeautifulSoup
from urllib.parse import urljoin

WEBSITE CONNECTION


In [6]:
BASE_URL = "https://books.toscrape.com/"
GBP_TO_INR = 105.50

WEB SCRAPING

In [7]:
def scrape_book(book, base_url):

    # Title
    title_link = book.find("h3").find("a")
    title = title_link["title"]

    # Price
    price = book.find(
        "p", class_="price_color"
    ).get_text(strip=True)

    # Rating
    rating_element = book.find(
        "p", class_="star-rating"
    )
    rating = rating_element.get("class")[1]

    # Availability
    availability = book.find(
        "p", class_="availability"
    ).get_text(" ", strip=True)

    # Book detail URL
    book_url = urljoin(base_url, title_link["href"])

    # Visit detail page
    detail_response = requests.get(book_url)
    detail_response.encoding = detail_response.apparent_encoding

    detail_soup = BeautifulSoup(
        detail_response.text,
        "html.parser"
    )

    # Category
    breadcrumb = detail_soup.find(
        "ul", class_="breadcrumb"
    )

    if breadcrumb is not None:
        breadcrumb_links = breadcrumb.find_all("a")

        if breadcrumb_links:
            category = breadcrumb_links[-1].get_text(
                strip=True
            )
        else:
            category = "Unknown"
    else:
        category = "Unknown"

    return {
        "title": title,
        "price": price,
        "rating": rating,
        "availability": availability,
        "category": category
    }

PAGINATION

In [8]:
all_books = []

for page_number in range(1, 5):

    if page_number == 1:
        page_url = BASE_URL
    else:
        page_url = urljoin(
            BASE_URL,
            f"catalogue/page-{page_number}.html"
        )

    print(f"Scraping page {page_number}...")

    response = requests.get(page_url)
    response.encoding = response.apparent_encoding

    page_soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    books = page_soup.find_all("article")

    for book in books:
        book_data = scrape_book(book, BASE_URL)
        all_books.append(book_data)
        time.sleep(0.2)

print("Total books scraped:", len(all_books))

Scraping page 1...
Scraping page 2...
Scraping page 3...
Scraping page 4...
Total books scraped: 80


CONVERT TO PANDAS DataFrame

In [9]:
df = pd.DataFrame(all_books)
df.to_csv("books_raw.csv", index=False)

DATA CLEANING

In [10]:
df["price_gbp"] = (
    df["price"]
    .str.replace("£", "", regex=False)
    .astype(float)
)

CLEAN RATING

In [11]:
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

df["rating"] = df["rating"].map(rating_map)


CLEAN AVAILABILITY

In [12]:
df["in_stock"] = (
    df["availability"]
    .str.contains("In stock")
)

GBP --> INR

In [13]:
GBP_TO_INR = 105.50

df["price_inr"] = (
    df["price_gbp"] * GBP_TO_INR
).round(2)
df = df[
    [
        "title",
        "category",
        "price_gbp",
        "price_inr",
        "rating",
        "in_stock"
    ]
]

DATA VALIDATION

In [14]:
print("Number of books:", len(df))
print("Number of categories:", df["category"].nunique())
print("Missing values:")
print(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Rating range:",
      df["rating"].min(),
      "to",
      df["rating"].max())

Number of books: 80
Number of categories: 14
Missing values:
title        0
category     0
price_gbp    0
price_inr    0
rating       0
in_stock     0
dtype: int64
Duplicate rows: 0
Rating range: 1 to 5


SAVING CLEANED DATASET

In [15]:
df.to_csv("books_cleaned.csv", index=False)

SQLite DATABASE

In [16]:
import os

if os.path.exists("books.db"):
    os.remove("books.db")
    import sqlite3

conn = sqlite3.connect("books.db")
cursor = conn.cursor()

CATEGORIES TABLE


In [17]:
cursor.execute("""
CREATE TABLE categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT UNIQUE NOT NULL
)
""")

In [18]:
categories = df["category"].unique()

for category in categories:
    cursor.execute(
        "INSERT INTO categories (category_name) VALUES (?)",
        (category,)
    )

conn.commit()

BOOKS TABLE

In [19]:
cursor.execute("""
CREATE TABLE books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    category_id INTEGER,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    in_stock BOOLEAN,
    FOREIGN KEY (category_id)
        REFERENCES categories(category_id)
)
""")

In [20]:
for _, row in df.iterrows():

    cursor.execute(
        "SELECT category_id FROM categories WHERE category_name = ?",
        (row["category"],)
    )

    category_id = cursor.fetchone()[0]

    cursor.execute("""
        INSERT INTO books (
            title,
            category_id,
            price_gbp,
            price_inr,
            rating,
            in_stock
        )
        VALUES (?, ?, ?, ?, ?, ?)
    """, (
        row["title"],
        category_id,
        row["price_gbp"],
        row["price_inr"],
        row["rating"],
        row["in_stock"]
    ))

conn.commit()

VERIFYING DATABASE

In [21]:
books_count = pd.read_sql(
    "SELECT COUNT(*) AS count FROM books",
    conn
)

categories_count = pd.read_sql(
    "SELECT COUNT(*) AS count FROM categories",
    conn
)

print("Books:", books_count["count"][0])
print("Categories:", categories_count["count"][0])

Books: 80
Categories: 14


 **SQL ANALYSIS**

QUERY 1 -AVG PRICE BY CATEGORY

In [22]:
query1 = """
SELECT
    c.category_name,
    ROUND(AVG(b.price_inr), 2) AS average_price_inr
FROM books b
JOIN categories c
    ON b.category_id = c.category_id
GROUP BY c.category_name
ORDER BY average_price_inr DESC;
"""

result1 = pd.read_sql(query1, conn)

result1

,category_name,average_price_inr
0,History,5721.26
1,Historical Fiction,5669.57
2,Sequential Art,5516.60
3,Politics,5415.32
4,Fiction,5285.55
5,Mystery,5045.01
6,Music,4867.24
7,Travel,4765.44
8,Science Fiction,3965.75
9,Poetry,3915.63


COUNT OF BOOKS BY RATING

In [23]:
query2 = """
SELECT
    rating,
    COUNT(*) AS book_count
FROM books
GROUP BY rating
ORDER BY rating;
"""

result2 = pd.read_sql(query2, conn)

result2

,rating,book_count
0,1,18
1,2,12
2,3,19
3,4,15
4,5,16


TOP 10 EXPENSIVE BOOKS

In [24]:
query3 = """
SELECT
    title,
    price_inr,
    rating
FROM books
ORDER BY price_inr DESC
LIMIT 10;
"""

result3 = pd.read_sql(query3, conn)

result3

,title,price_inr,rating
0,The Death of Humanity: and the Case for Life,6130.60,4
1,Slow States of Collapse: Poems,6046.20,3
2,Our Band Could Be Your Life: Scenes from the A...,6039.88,3
3,The Past Never Ends,5960.75,4
4,The Pioneer Woman Cooks: Dinnertime: Comfort C...,5951.25,1
5,The Secret of Dreadwillow Carse,5921.72,1
6,The Electric Pencil: Drawings from Inside Stat...,5914.33,1
7,Birdsong: A Story in Pictures,5764.52,3
8,Sapiens: A Brief History of Humankind,5721.26,5
9,The Murder That Never Was (Forensic Instincts #5),5708.60,3


BOOKS ABOVE AVG PRICE

In [25]:
query4 = """
SELECT
    title,
    price_inr
FROM books
WHERE price_inr > (
    SELECT AVG(price_inr)
    FROM books
)
ORDER BY price_inr DESC;
"""

result4 = pd.read_sql(query4, conn)

result4

,title,price_inr
0,The Death of Humanity: and the Case for Life,6130.60
1,Slow States of Collapse: Poems,6046.20
2,Our Band Could Be Your Life: Scenes from the A...,6039.88
3,The Past Never Ends,5960.75
4,The Pioneer Woman Cooks: Dinnertime: Comfort C...,5951.25
5,The Secret of Dreadwillow Carse,5921.72
6,The Electric Pencil: Drawings from Inside Stat...,5914.33
7,Birdsong: A Story in Pictures,5764.52
8,Sapiens: A Brief History of Humankind,5721.26
9,The Murder That Never Was (Forensic Instincts #5),5708.60


STOCK AVAILABILITY BY CATEGORY

In [26]:
query5 = """
SELECT
    c.category_name,
    SUM(CASE WHEN b.in_stock = 1 THEN 1 ELSE 0 END) AS in_stock,
    SUM(CASE WHEN b.in_stock = 0 THEN 1 ELSE 0 END) AS out_of_stock
FROM books b
JOIN categories c
    ON b.category_id = c.category_id
GROUP BY c.category_name
ORDER BY c.category_name;
"""

result5 = pd.read_sql(query5, conn)

result5

,category_name,in_stock,out_of_stock
0,Business,1,0
1,Default,3,0
2,Fiction,1,0
3,Historical Fiction,1,0
4,History,1,0
5,Music,2,0
6,Mystery,1,0
7,Poetry,4,0
8,Politics,1,0
9,Science Fiction,1,0


PANDAS ANALYSIS

In [27]:
books_df = pd.read_sql(
    "SELECT * FROM books;",
    conn
)
categories_df = pd.read_sql(
    "SELECT * FROM categories;",
    conn
)
books_df.head()


,book_id,title,category_id,price_gbp,price_inr,rating,in_stock
0,1,A Light in the Attic,1,51.77,5461.74,3,1
1,2,Tipping the Velvet,2,53.74,5669.57,1,1
2,3,Soumission,3,50.10,5285.55,1,1
3,4,Sharp Objects,4,47.82,5045.01,4,1
4,5,Sapiens: A Brief History of Humankind,5,54.23,5721.26,5,1


MERGED BOOKS & CATEGORIES

In [28]:
merged_df = pd.merge(
    books_df,
    categories_df,
    on="category_id",
    how="left"
)

merged_df.head()

,book_id,title,category_id,price_gbp,price_inr,rating,in_stock,category_name
0,1,A Light in the Attic,1,51.77,5461.74,3,1,Poetry
1,2,Tipping the Velvet,2,53.74,5669.57,1,1,Historical Fiction
2,3,Soumission,3,50.10,5285.55,1,1,Fiction
3,4,Sharp Objects,4,47.82,5045.01,4,1,Mystery
4,5,Sapiens: A Brief History of Humankind,5,54.23,5721.26,5,1,History


CATEGORY WISE AVG PRICE

In [29]:
category_analysis = (
    merged_df
    .groupby("category_name")["price_inr"]
    .agg(["count", "mean", "min", "max"])
    .round(2)
    .sort_values("mean", ascending=False)
)

category_analysis

,count,mean,min,max
category_name,,,,
History,1,5721.26,5721.26,5721.26
Historical Fiction,1,5669.57,5669.57,5669.57
Sequential Art,1,5516.60,5516.60,5516.60
Politics,1,5415.32,5415.32,5415.32
Fiction,1,5285.55,5285.55,5285.55
Mystery,1,5045.01,5045.01,5045.01
Music,2,4867.24,3694.61,6039.88
Travel,1,4765.44,4765.44,4765.44
Science Fiction,1,3965.75,3965.75,3965.75


RATING DISTRIBUTION

In [30]:
rating_analysis = (
    merged_df["rating"]
    .value_counts()
    .sort_index()
)

rating_analysis

,count
rating,
1,18
2,12
3,19
4,15
5,16


TOP 10 EXPENSIVE BOOKS

In [31]:
top_10_books = (
    merged_df[
        ["title", "category_name", "price_gbp", "price_inr", "rating"]
    ]
    .sort_values("price_inr", ascending=False)
    .head(10)
)

top_10_books

,title,category_name,price_gbp,price_inr,rating
68,The Death of Humanity: and the Case for Life,Unknown,58.11,6130.60,4
40,Slow States of Collapse: Poems,Unknown,57.31,6046.20,3
15,Our Band Could Be Your Life: Scenes from the A...,Music,57.25,6039.88,3
58,The Past Never Ends,Unknown,56.50,5960.75,4
57,The Pioneer Woman Cooks: Dinnertime: Comfort C...,Unknown,56.41,5951.25,1
56,The Secret of Dreadwillow Carse,Unknown,56.13,5921.72,1
67,The Electric Pencil: Drawings from Inside Stat...,Unknown,56.06,5914.33,1
25,Birdsong: A Story in Pictures,Unknown,54.64,5764.52,3
4,Sapiens: A Brief History of Humankind,History,54.23,5721.26,5
61,The Murder That Never Was (Forensic Instincts #5),Unknown,54.11,5708.60,3


STOCK ANALYSIS

In [32]:
stock_analysis = (
    merged_df
    .groupby("category_name")["in_stock"]
    .agg(["count", "sum"])
    .rename(
        columns={
            "count": "total_books",
            "sum": "in_stock_books"
        }
    )
)

stock_analysis["out_of_stock_books"] = (
    stock_analysis["total_books"]
    - stock_analysis["in_stock_books"]
)

stock_analysis

,total_books,in_stock_books,out_of_stock_books
category_name,,,
Business,1,1,0
Default,3,3,0
Fiction,1,1,0
Historical Fiction,1,1,0
History,1,1,0
Music,2,2,0
Mystery,1,1,0
Poetry,4,4,0
Politics,1,1,0


OVERALL STATISTICS

In [33]:
summary = {
    "Total Books": len(merged_df),
    "Total Categories": merged_df["category_name"].nunique(),
    "Average Price GBP": round(merged_df["price_gbp"].mean(), 2),
    "Average Price INR": round(merged_df["price_inr"].mean(), 2),
    "Average Rating": round(merged_df["rating"].mean(), 2),
    "Books In Stock": int(merged_df["in_stock"].sum())
}

summary

{'Total Books': 80,
 'Total Categories': 14,
 'Average Price GBP': np.float64(35.71),
 'Average Price INR': np.float64(3767.23),
 'Average Rating': np.float64(2.99),
 'Books In Stock': 80}

SAVING FINAL CLEANED DATASET

In [34]:
final_df = merged_df[
    [
        "book_id",
        "title",
        "category_id",
        "category_name",
        "price_gbp",
        "price_inr",
        "rating",
        "in_stock"
    ]
]

final_df.to_csv(
    "books_final.csv",
    index=False
)

In [35]:
final_df.head(10)


,book_id,title,category_id,category_name,price_gbp,price_inr,rating,in_stock
0,1,A Light in the Attic,1,Poetry,51.77,5461.74,3,1
1,2,Tipping the Velvet,2,Historical Fiction,53.74,5669.57,1,1
2,3,Soumission,3,Fiction,50.10,5285.55,1,1
3,4,Sharp Objects,4,Mystery,47.82,5045.01,4,1
4,5,Sapiens: A Brief History of Humankind,5,History,54.23,5721.26,5,1
5,6,The Requiem Red,6,Young Adult,22.65,2389.57,1,1
6,7,The Dirty Little Secrets of Getting Your Dream...,7,Business,33.34,3517.37,4,1
7,8,The Coming Woman: A Novel Based on the Life of...,8,Default,17.93,1891.62,3,1
8,9,The Boys in the Boat: Nine Americans and Their...,8,Default,22.60,2384.30,4,1
9,10,The Black Maria,1,Poetry,52.15,5501.82,1,1
